# Milestone 2: ED Solver Comparison

- Compare ED vs IPT at the same parameters
- Study convergence with increasing bath size M_g

In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from dmft.config import DMFTParams
from dmft.solvers.ipt import IPTSolver
from dmft.solvers.ed import EDSolver
from dmft.dmft_loop import dmft_loop
from dmft.observables import spectral_function_from_poles

## 1. IPT vs ED at U=2

In [ ]:
U = 2.0
beta = 50.0

# IPT
params_ipt = DMFTParams.half_filling(U=U, beta=beta, n_matsubara=512, M_g=2, M_h=2)
params_ipt.max_iter = 80; params_ipt.mix = 0.3
result_ipt = dmft_loop(params_ipt, IPTSolver(), verbose=False)
print(f'IPT:  Z={result_ipt["Z"]:.4f}, n={result_ipt["n_imp"]:.4f}')

# ED
params_ed = DMFTParams.half_filling(U=U, beta=beta, n_matsubara=256, M_g=2, M_h=2)
params_ed.max_iter = 60; params_ed.mix = 0.3
result_ed = dmft_loop(params_ed, EDSolver(), verbose=False)
print(f'ED:   Z={result_ed["Z"]:.4f}, n={result_ed["n_imp"]:.4f}')

In [ ]:
# Compare Green's functions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
wn_ipt = result_ipt['wn']; wn_ed = result_ed['wn']

ax1.plot(wn_ipt[:30], result_ipt['G_loc'][:30].imag, 'b.-', ms=3, label='IPT')
ax1.plot(wn_ed[:30], result_ed['G_loc'][:30].imag, 'r.-', ms=3, label='ED')
ax1.set_xlabel(r'$\omega_n$'); ax1.set_ylabel(r'Im $G_{loc}$')
ax1.legend(); ax1.set_title(f'U={U}')

ax2.plot(wn_ipt[:30], result_ipt['Sigma'][:30].imag, 'b.-', ms=3, label='IPT')
ax2.plot(wn_ed[:30], result_ed['Sigma'][:30].imag, 'r.-', ms=3, label='ED')
ax2.set_xlabel(r'$\omega_n$'); ax2.set_ylabel(r'Im $\Sigma$')
ax2.legend(); ax2.set_title(f'Self-energy, U={U}')
plt.tight_layout(); plt.show()

## 2. Bath size convergence (M_g = 1, 2, 3)

In [ ]:
omega = np.linspace(-4, 4, 2000)
M_values = [1, 2, 3]
results_M = {}

for M in M_values:
    p = DMFTParams.half_filling(U=U, beta=beta, n_matsubara=256, M_g=M, M_h=M)
    p.max_iter = 60; p.mix = 0.3
    r = dmft_loop(p, EDSolver(), verbose=False)
    results_M[M] = r
    print(f'M_g={M}: Z={r["Z"]:.4f}, n={r["n_imp"]:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
for M in M_values:
    r = results_M[M]
    p = r['poles']
    A = spectral_function_from_poles(omega, U/2, 0.0, p.V, p.eps, p.W, p.eta, p.sigma_inf, 0.05)
    ax.plot(omega, A, label=f'$M_g$={M}, Z={r["Z"]:.3f}')

ax.set_xlabel(r'$\omega$'); ax.set_ylabel(r'$A(\omega)$')
ax.legend(); ax.set_title(f'ED: bath size convergence, U={U}')
plt.tight_layout(); plt.show()